# Quantum Machine Learning from primitives

A quantum machine learning model is a parameterized quantum circuit whose parameters are tuned to fit data. In this tutorial we build one from the ground up out of the primitives in `qrisp.qml`, train it, and then vary each part in turn.

There is no model class to instantiate and no training loop to inherit. A model here is an ordinary Python function, the loss is an ordinary Python function, and training is a call to `qrisp.jasp.minimize`. We assemble everything explicitly, which means every part can be replaced.

The figure shows the whole pipeline for the model we build here, with two features on two qubits. Throughout, we write vectors underlined, such as the features $\underline{x} = (x_0, x_1)$ of one sample and the parameters $\underline{w}$, and their components in plain italic, such as $x_0$:

<p align="center"><img src="../../_static/qml_pipeline.svg" alt="The QML pipeline: the features x enter the encoding S(x), the ansatz W(w) acts on the encoded state, qubit 0 is measured in Z to give the prediction f(x; w), the loss compares predictions with labels, and the optimizer updates w." width="800"></p>

Read it from left to right.

- **Encoding $S(\underline{x})$.** A quantum circuit cannot take numbers as input directly, so the first step writes the features into the qubits. Each feature becomes the angle of a rotation $R_Y(x_i)$, which turns the starting state $\ket{00}$ into a state $\ket{\psi(\underline{x})}$ that depends on the data.
- **Ansatz $W(\underline{w})$.** Next comes a circuit of rotations with adjustable angles $\underline{w}$, and a CX gate that lets the two qubits influence each other. The ansatz plays the role of the weights in a neural network: the data decide where the state starts, and $\underline{w}$ decides how it is transformed into $\ket{\psi(\underline{x}, \underline{w})}$.
- **Measurement.** A quantum state cannot be read out directly. We measure qubit 0 many times and average the results, which gives the expectation value $f(\underline{x}; \underline{w}) = \langle \psi(\underline{x}, \underline{w}) | Z_0 | \psi(\underline{x}, \underline{w}) \rangle$, a number between $-1$ and $+1$. This number is the model's prediction.
- **Loss.** The loss $L(\underline{w})$ measures how far the predictions are from the true labels $y^{(i)}$, averaged over the training data. A perfect model has loss $0$.
- **Optimizer.** The optimizer tries new values of $\underline{w}$, keeps those that lower the loss, and repeats. This loop, the dashed arrow, is training. Only $\underline{w}$ changes; the encoding stays fixed.

Once training is done, the loop is no longer needed: the trained model is the first three stages alone, with $\underline{w}$ fixed at the values the optimizer found.

## Setup

Besides NumPy, JAX and Matplotlib, we use four parts of Qrisp:

- `qrisp` provides `QuantumFloat`, the quantum register our circuits act on. We use it only as a plain register of qubits, not for the numbers it can represent.
- `qrisp.operators` provides the Pauli operators, such as `Z`, that we use for measurement.
- `qrisp.jasp` is Qrisp's compilation layer. `jaspify` compiles a function that mixes quantum and classical code so that it runs fast, and `minimize` is the optimizer that trains the parameters.
- `qrisp.qml` holds the building blocks of this tutorial: `Encoding` and `Ansatz` for the circuit, `expectation` for the readout, `minmax_scale`, `normalize` and `binarize` for preparing data, `real_amplitudes_size` and `hardware_efficient_size` for counting an ansatz's parameters, and `save_model` and `load_model` for saving a trained model and loading it again.

In [ ]:
import tempfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from IPython.display import clear_output

from qrisp import QuantumFloat
from qrisp.operators import Z
from qrisp.jasp import jaspify, minimize
from qrisp.qml import (
    minmax_scale,
    normalize,
    binarize,
    Encoding,
    real_amplitudes_size,
    Ansatz,
    expectation,
    save_model,
    load_model,
    hardware_efficient_size,
)

## The problem

We use the [Iris dataset](https://archive.ics.uci.edu/dataset/53/iris): 150 flowers from three species, each described by four measurements in centimetres. We take the two species that are hardest to tell apart, *versicolor* (label $-1$) and *virginica* (label $+1$), and classify them by their petal length and petal width.

The two species are hard to tell apart by eye as well. Both are blue-violet irises with three large drooping sepals and three smaller upright petals, and it is the size of those parts, not their colour, that separates them.

![Iris versicolor](./iris_versicolor.jpg) ![Iris virginica](./iris_virginica.jpg)

*Iris versicolor* (left) and *Iris virginica* (right). Photos: [Danielle Langlois](https://commons.wikimedia.org/wiki/File:Iris_versicolor_3.jpg), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/); [Frank Mayfield](https://commons.wikimedia.org/wiki/File:Iris_virginica.jpg), [CC BY-SA 2.0](https://creativecommons.org/licenses/by-sa/2.0/); both via Wikimedia Commons, scaled down.

Training is deliberately small. Every evaluation of the loss runs the circuit once per training sample, and the optimizer evaluates the loss hundreds of times, so we train on four flowers of each species, picked to run from its shortest petal to its longest so that eight flowers still cover the range of sizes. We hold the other 92 flowers back as a test set: the model never sees them during training, so they show whether it learned the species or only memorised its eight training flowers.

## Loading the data

The UCI repository serves Iris as a plain comma-separated file with no header row. Each line is one flower: sepal length, sepal width, petal length and petal width in centimetres, then the species name. There are 50 flowers of each species.

Of the four measurements we use two: **petal length** and **petal width**. They carry most of the difference between the two species. A single cut on either one sorts 93–94% of the versicolor and virginica flowers correctly, against 73% for sepal length and 63% for sepal width. The sepals come back later, when we try an encoding that fits all four measurements on two qubits.

We also drop *setosa*, whose petals are so much smaller than the other two species' that a single threshold on petal length separates it, leaving nothing to learn. The cell below then labels the remaining 100 flowers and makes the split described above: it sorts each species by petal length and takes four evenly spaced flowers from that order for training.

In [ ]:
IRIS_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
rows = [line.split(",") for line in urlopen(IRIS_URL).read().decode().splitlines() if line]
measurements = np.array([[float(value) for value in row[:4]] for row in rows])
species = np.array([row[4] for row in rows])

keep = species != "Iris-setosa"
measurements, species = measurements[keep], species[keep]
labels = np.where(species == "Iris-versicolor", -1.0, 1.0)

by_length = [
    np.flatnonzero(labels == label)[np.argsort(measurements[labels == label, 2], kind="stable")]
    for label in (-1.0, 1.0)
]
train_idx = np.concatenate([order[np.linspace(0, len(order) - 1, 4).round().astype(int)] for order in by_length])
test_idx = np.setdiff1d(np.arange(len(labels)), train_idx)

# Columns 2 and 3 are petal length and petal width.
X_raw, X_test_raw = measurements[train_idx][:, 2:], measurements[test_idx][:, 2:]
y_train, y_test = jnp.asarray(labels[train_idx]), jnp.asarray(labels[test_idx])

print(f"training flowers: {len(train_idx)}, test flowers: {len(test_idx)}\n")

print("Data \nsl.no \t petal length \t petal width \t species")
for i in range(len(X_raw)):
    print(f"{i + 1} \t {X_raw[i, 0]:.1f} \t\t {X_raw[i, 1]:.1f} \t\t {y_train[i]}")

## Scaling

Our features are lengths in centimetres, but angle encoding reads each feature as a rotation angle in radians. It knows nothing about units: a 6.9 cm petal becomes a rotation by 6.9. Rotation angles repeat every full turn of $2\pi \approx 6.28$, so that petal lands in the same state as a 0.62 cm petal, and the circuit cannot tell the two apart. Nothing raises an error; the model simply learns worse.

<p align="center"><img src="../../_static/qml_angle_wrap.svg" alt="Two circles side by side. A 0.62 cm petal turns by 0.62 from the start; a 6.9 cm petal turns by 6.9, one full turn plus 0.62. Both end at the same point, so they encode to the same state." width="560"></p>

So before encoding, we scale every feature into a range shorter than a full turn.

`minmax_scale` maps each feature from its range on the *training* data onto $[0, \pi]$ by default. We pass the test set as a second argument, so the same two numbers are applied to it and no information about the test data reaches the model during training. A test value beyond the training range is clipped to the interval, which happens to a few test values here: eight training flowers cannot reach every extreme of the other 92.

In [ ]:
X_train, X_test = minmax_scale(X_raw, X_test_raw)

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.6))

colors = ["tab:blue" if label < 0 else "tab:red" for label in np.asarray(y_train)]
test_colors = ["tab:blue" if label < 0 else "tab:red" for label in np.asarray(y_test)]

for axes, train_points, test_points in [(left, X_raw, X_test_raw), (right, X_train, X_test)]:
    axes.scatter(
        test_points[:, 0], test_points[:, 1], facecolors="none", edgecolors=test_colors, s=40, alpha=0.6, label="test"
    )
    axes.scatter(train_points[:, 0], train_points[:, 1], c=colors, s=80, edgecolors="black", label="training")
    axes.set_xlabel("petal length")
    axes.set_ylabel("petal width")

left.set_title("raw features (cm)")
left.legend()
right.set_title("scaled to $[0, \\pi]$")
for axis in (right.set_xlim, right.set_ylim):
    axis(-0.2, np.pi + 0.2)

fig.tight_layout()
plt.show()

print("raw    range per feature:", X_raw.min(axis=0), "to", X_raw.max(axis=0))
print("scaled range per feature:", np.round(X_train.min(axis=0), 3), "to", np.round(X_train.max(axis=0), 3))
print("test values clipped to the interval:", int(np.sum((X_test == 0.0) | (X_test == np.pi))))

# The scaled features go on to the quantum side, so hand them over as jax arrays.
X_train = jnp.asarray(X_train)
X_test = jnp.asarray(X_test)

The shape of the data is unchanged; only the axes are. That is the point: the species stay as separable as they were, but every value is now a usable rotation angle. They are not perfectly separable: a few versicolor and virginica flowers overlap, so no classifier can be expected to get every test flower right.

Two further helpers cover the other encodings' input requirements. `normalize` scales each sample to unit norm, which is what `Encoding.amplitude` needs, and `binarize` reduces features to the 0/1 values `Encoding.basis` works in.

In [ ]:
print("normalize :", np.round(normalize(np.array([[3.0, 4.0]])), 3), " (norm 5 -> 1)")
print("binarize  :", binarize(np.array([[0.8, -0.2, 1.5]])))

## Encoding

A quantum circuit cannot read numbers. It always starts from the same state, which in our 2 qubit case here is $\ket{00}$, so the only way data can get in is through gates whose settings depend on the data. How we choose those gates is the **encoding**, and it matters more than it first appears. Everything the model computes afterwards starts from the encoded state, so the encoding decides what the model *can* learn. Two flowers that encode to the same state, like the unscaled petals above, can never be told apart, whatever the ansatz does. The encoding also fixes which *kind* of function of the features the model can produce, as we will see near the end of this tutorial.

### Angle encoding

`Encoding.angle` writes each feature into the rotation angle of one qubit, so a register of $n$ qubits holds $n$ features. To see what that means, start with a single qubit. The rotation $R_Y(\theta)$ turns $\ket{0}$ into

$$R_Y(\theta)\ket{0} = \cos(\theta/2)\ket{0} + \sin(\theta/2)\ket{1}.$$

The two numbers in front of $\ket{0}$ and $\ket{1}$ are amplitudes; squared, they give the probabilities of measuring $0$ and $1$. At $\theta = 0$ the qubit stays $\ket{0}$. At $\theta = \pi$, $\cos(\pi/2) = 0$ and $\sin(\pi/2) = 1$, so the qubit is in $\ket{1}$. In between, the probability of measuring $1$, $\sin^2(\theta/2)$, rises smoothly from $0$ to $1$. On the Bloch sphere, this is the state turning by $\theta$ from $\ket{0}$ at the top towards $\ket{1}$ at the bottom:

<p align="center"><img src="../../_static/qml_ry_rotation.svg" alt="A slice through the Bloch sphere with |0> at the top and |1> at the bottom. The state R_Y(theta)|0> points at angle theta from the top; its height is <Z> = cos(theta). Theta = 0 gives |0>, theta = pi/2 an equal superposition, theta = pi gives |1>. The state is cos(theta/2)|0> + sin(theta/2)|1>, and the probability of measuring 1 is sin^2(theta/2)." width="640"></p>

Angle encoding simply uses each feature as the angle: qubit $0$ is rotated by $x_0$ and qubit $1$ by $x_1$. The two qubits are rotated independently, so the register holds the two one-qubit states side by side, written with $\otimes$:

$$\big(\cos(x_0/2)\ket{0} + \sin(x_0/2)\ket{1}\big) \otimes \big(\cos(x_1/2)\ket{0} + \sin(x_1/2)\ket{1}\big).$$

The symbol $\otimes$ (the tensor product) just means "qubit $0$ in this state, qubit $1$ in that one". Multiplying it out like ordinary brackets gives one term for every combination of the two qubits, with their amplitudes multiplied. Writing qubit $0$ first in each ket:

$$\cos\tfrac{x_0}{2}\cos\tfrac{x_1}{2}\,\ket{00} + \cos\tfrac{x_0}{2}\sin\tfrac{x_1}{2}\,\ket{01} + \sin\tfrac{x_0}{2}\cos\tfrac{x_1}{2}\,\ket{10} + \sin\tfrac{x_0}{2}\sin\tfrac{x_1}{2}\,\ket{11}.$$

Squaring an amplitude again gives the probability of that outcome. Both qubits read $1$, that is the system is in state $\ket{11}$, with probability $\sin^2(x_0/2)\,\sin^2(x_1/2)$, the product of the two single-qubit probabilities: so far, each qubit carries its own feature independently.

This is why we scaled into $[0, \pi]$: the smallest training value lands on $\ket{0}$, the largest on $\ket{1}$, and every value in between gets its own state, with nothing wrapping round. After scaling, versicolor with its smaller petals sits towards $\ket{00}$ and virginica towards $\ket{11}$.

### Other encodings

Angle encoding is the simplest choice, not the only one. `qrisp.qml` provides four encodings, and they trade the number of qubits against the cost of the circuit and what they keep of the data:

| Encoding | Features on $n$ qubits | How a feature enters | What it needs |
|---|---|---|---|
| `Encoding.angle` | $n$ | one rotation per qubit | features scaled to angles |
| `Encoding.dense_angle` | $2n$ | a rotation and a relative phase per qubit | features scaled to angles |
| `Encoding.amplitude` | $2^n$ | as the amplitudes of the whole register | unit length, which it enforces itself, so only the direction of the feature vector survives |
| `Encoding.basis` | $n$ | each qubit set to $0$ or $1$ | bits; other values are reduced to bits |

Packing more features into fewer qubits is not free. Amplitude encoding fits exponentially many features, but preparing an arbitrary state takes a deep circuit, around two orders of magnitude slower per call than angle encoding. Basis encoding suits data that really is binary and loses everything else: with the default threshold, every petal length becomes $1$. Dense angle encoding halves the qubit count, at the price of two features sharing one qubit.

For two features on two qubits, angle encoding is the natural choice: one cheap rotation per feature, and a state we can reason about directly. We try `dense_angle` and `amplitude` later in this tutorial.

**Further reading.** M. Schuld and F. Petruccione, [*Machine Learning with Quantum Computers*](https://doi.org/10.1007/978-3-030-83098-4), 2nd ed., Springer (2021), is the standard textbook on the field and covers how classical data is encoded into quantum states. R. LaRose and B. Coyle, [*Robust data encodings for quantum classifiers*](https://doi.org/10.1103/PhysRevA.102.032420), Phys. Rev. A **102**, 032420 (2020) ([arXiv:2003.01695](https://arxiv.org/abs/2003.01695)), defines angle, dense angle and amplitude encoding (there called wavefunction encoding) and shows that the encoding decides which decision boundaries a classifier can learn. M. Schuld, R. Sweke and J. J. Meyer, [*Effect of data encoding on the expressive power of variational quantum machine learning models*](https://doi.org/10.1103/PhysRevA.103.032430), Phys. Rev. A **103**, 032430 (2021) ([arXiv:2008.08605](https://arxiv.org/abs/2008.08605)), explains why the encoding fixes the kind of function a model can express, which we meet in the section on a curve one encoding layer cannot fit.

Let us encode three of our training flowers: the shortest versicolor, the longest versicolor, and the longest virginica. For each one we print the probability of every outcome when both qubits are measured.

Qrisp prints a register as a number rather than as a string of bits, with qubit $0$ as the lowest bit, so $\ket{00}$ appears as `|0>` and $\ket{11}$ as `|3>`. The small helper below converts the outcomes back into bits, written with qubit $0$ first as in the formula above.

In [ ]:
encoding = Encoding.angle(n_features=2)


def outcome_probabilities(qf):
    # Qrisp reports the register as a number with qubit 0 as the lowest bit; spell it out with qubit 0 first.
    return {f"{int(v) & 1}{int(v) >> 1}": round(float(p), 3) for v, p in sorted(qf.get_measurement().items())}


for name, i in [("shortest versicolor", 0), ("longest versicolor", 3), ("longest virginica", 7)]:
    qf = QuantumFloat(2)
    encoding.apply(qf, X_train[i])
    print(f"{name:20s} features {np.round(np.asarray(X_train[i]), 2)} -> {outcome_probabilities(qf)}")

Each qubit holds one feature: qubit $0$ the petal length, qubit $1$ the petal width. A measured $1$ on a qubit loosely means "long" or "wide", and it becomes more likely the further along its scaled range the feature sits, with probability $\sin^2(x_i/2)$. So `00` reads as short and narrow, `11` as long and wide. Because the two qubits are encoded independently, each outcome's probability is the product of the two single-qubit probabilities.

The shortest versicolor sits at the bottom of both ranges and gives `00` almost every time; the longest virginica sits at the top and gives `11`. The longest versicolor lies in the middle of both ranges, where the species overlap, and its outcomes are spread across all four.

Each qubit on its own is a slice through its Bloch sphere, exactly as in the single-qubit figure above: the arrow sits at the angle of its feature, and its probability of reading $1$ grows as it turns from $\ket{0}$ towards $\ket{1}$. This one-picture-per-qubit view works only while the qubits are independent, as they are right after encoding; once the ansatz entangles them, a qubit no longer has a state of its own.

<p align="center"><img src="../../_static/qml_encoding_bloch.svg" alt="Three rows, one per flower, each with two Bloch-circle slices: qubit 0 for petal length and qubit 1 for petal width. A, the shortest versicolor: both arrows point near |0>, x0 = 0 (P(1) = 0%) and x1 = 0.21 (P(1) = 1%). B, the longest versicolor: x0 = 1.69 (P(1) = 56%) and x1 = 1.26 (P(1) = 35%), both arrows near the equator. C, the longest virginica: x0 = pi (P(1) = 100%) and x1 = 2.72 (P(1) = 96%), both arrows near |1>. A bar chart beside each row gives the probabilities of the four outcomes when both qubits are measured: A 99% 00 and 1% 01; B 29% 00, 37% 10, 15% 01, 19% 11; C 96% 11 and 4% 10." width="800"></p>

The bar charts on the right show what happens when both qubits are measured together: each outcome's probability is the product of the two qubits' own probabilities. Flower B, for example, gives `10`, long and narrow, with probability $0.56 \times 0.65 \approx 0.37$.

## Ansatz

The encoding gives the model no freedom: the same flower always becomes the same state. To learn, the circuit also needs settings we can change. That is the job of the **ansatz**, a circuit whose rotation angles are the model's parameters $\underline{w}$. During training, the optimizer adjusts these angles, much like the weights of a neural network. `Ansatz.real_amplitudes` alternates layers of $R_Y$ rotations with CX entanglers:

$$\left( \prod_{l=1}^{L} U_{\text{ent}} R_Y(\underline{w}^{(l)}) \right) R_Y(\underline{w}^{(0)})$$

Here $R_Y(\underline{w}^{(l)})$ is one layer of rotations, one per qubit, whose angles are the part $\underline{w}^{(l)}$ of the parameter vector; in the figure below, the first layer has the angles $w_0$ and $w_1$, the second $w_2$ and $w_3$.

There is one more rotation layer than entangling layers, so the output of the last entangler is rotated before it is measured.

Here is the circuit we build, with the encoding and the ansatz marked:

<p align="center"><img src="../../_static/qml_circuit.svg" alt="The two-qubit circuit. Both qubits start in |0>. The encoding S(x), marked in green, rotates qubit 0 by R_Y(x0) and qubit 1 by R_Y(x1), using the features x = (x0, x1), and produces the state |psi(x)>. The ansatz W(w), marked in blue, applies R_Y(w0) and R_Y(w1), a CX gate with qubit 0 as control and qubit 1 as target, then R_Y(w2) and R_Y(w3), and produces the state |psi(x, w)>." width="560"></p>

Every ansatz has a companion size function giving the number of parameters it needs. We use it rather than working the count out by hand, because it keeps the parameter vector correct when the register size or layer count changes.

Below we build the ansatz, apply it with untrained parameters to just one sample after encoding it, and print the circuit Qrisp has built: first the two encoding rotations, then the ansatz.

In [ ]:
num_qubits = 2
layers = 1
num_params = real_amplitudes_size(num_qubits, layers=layers)
ansatz = Ansatz.real_amplitudes(num_qubits=num_qubits, layers=layers)
print(f"{num_qubits} qubits, {layers} layer(s) -> {num_params} parameters\n")

# Untrained parameters, chosen distinct so each gate's angle is visible in the drawing.
w_example = jnp.array([0.5, 1.0, 1.5, 2.0])

qf = QuantumFloat(num_qubits)
encoding.apply(qf, X_train[0])
ansatz.apply(qf, w_example)
print(qf.qs)

## Measurement

To get a prediction out of the circuit, we measure it. A measurement is always made along some axis; the usual choice, and ours here, is the $Z$ axis. Measured along $Z$, a qubit gives one of two outcomes, $\ket{0}$ or $\ket{1}$. A single measurement tells us very little, so we prepare and measure the circuit many times and count how often each outcome appears. This gives the probability distribution of the outcomes. (It is not the full state: phases, for example, do not show up in these counts.)

For a classifier we want a single number, so we score each outcome: $+1$ for $\ket{0}$ and $-1$ for $\ket{1}$, which matches the $\pm 1$ labels. The average score over many runs is the **expectation value** $\langle Z_0 \rangle$; in code we ask for it with the observable `Z(0)`. The function `expectation`, provided by the `qrisp.qml` module, does the repeated measuring and averaging for us. Because it has to run the circuit many times, we pass it a function that builds the circuit, `prepare` below, rather than a circuit that has already been built.

We read out qubit 0 alone. Angle encoding needs two qubits to hold the two features, though an encoding such as `Encoding.dense_angle`, which we try later, could fit both into one; a binary classifier needs only one output number either way. Qubit 1 still influences the prediction: the CX gate entangles the two qubits, and the final rotation $R_Y(w_2)$ on qubit 0 turns that entanglement into a change in $\langle Z_0 \rangle$. The flip side is that a gate on qubit 1 *after* the CX can no longer reach the measured qubit, so the last rotation $R_Y(w_3)$ has no effect on the prediction: with this readout, only three of the four parameters matter. Whenever part of a register is left unmeasured, it is worth checking which gates can still affect what is read out.

Below we read out $\langle Z_0 \rangle$ for the encoded flower from the previous cell. `precision` sets how tightly each expectation value is estimated. The shot count scales with the inverse square of it, so `0.05` is far cheaper than the default `0.01` and accurate enough while iterating.

In [ ]:
def prepare():
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, X_train[0])
    ansatz.apply(qf, w_example)
    return qf


value = expectation(prepare, Z(0), precision=0.05)()
print(f"Sample 0(shortest versicolor), untrained parameters: <Z_0> = {float(value):+.3f}")

## The model

Encoding, ansatz and measurement together are the model: a function that takes a flower's features $\underline{x}$ and the parameters $\underline{w}$ and returns a prediction. So far we ran each stage by hand on a single flower. The functions `circuit` and `model` below package the same three steps, so that we can apply them to any flower, which is what training needs.

With untrained parameters, the model gives a versicolor and a virginica almost the same output: it does not yet tell the species apart. Finding parameters that push versicolor towards $-1$ and virginica towards $+1$ is the job of training.

In [ ]:
def circuit(x, w):
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, x)
    ansatz.apply(qf, w)
    return qf


def model(x, w):
    return expectation(lambda: circuit(x, w), Z(0), precision=0.05)()


for name, i in [("shortest versicolor", 0), ("longest virginica", 7)]:
    print(f"{name:20s} label {float(y_train[i]):+.0f}  ->  untrained prediction {float(model(X_train[i], w_example)):+.3f}")

## Loss and training

For the loss we use the mean squared error between predictions and labels. It takes only the parameters, which is the signature `minimize` expects. `make_loss` builds that function for a given model and dataset, so we can reuse it with the other models later on.

`minimize` is **gradient-free**. It samples the loss rather than differentiating it, because the expectation values come from sampling and carry no derivative. Tracing a circuit with Jasp makes it fast, because the circuit is built once and then reused instead of being rebuilt in Python for every evaluation, but it does not make it differentiable. Gradients can still be obtained from extra circuit runs with the parameter-shift rule; the [transfer learning tutorial](QMLTransferLearning.ipynb) trains a model that way.

In [ ]:
def make_loss(model_fn, X, y):
    def loss(w):
        total = 0.0
        for i in range(X.shape[0]):
            total = total + jnp.sum((model_fn(X[i], w) - y[i]) ** 2)
        return total / X.shape[0]

    return loss

We run training inside `@jaspify`. Jasp owns the outermost layer of a hybrid program: quantum and classical code compose freely *inside* it, but it cannot be nested inside `jax.jit` from the outside.

The optimizer is **local**: it descends from its starting point into the nearest minimum. A run that stalls at a poor loss is therefore not evidence that the model cannot fit the data; it usually means the starting point was bad. In practice one trains from several starting points and keeps the best. Here we keep things simple and use a single starting point.

`minimize` only returns the final result. To watch training as it happens, we let the loss report every value it computes, using `jax.debug.callback`, and redraw a plot after each iteration of the optimizer, showing the lowest loss among the parameter values it keeps.

We set `maxiter` to a reasonable value (20). Internally the `minimize` function keeps a step size, called `rho`, which starts at $1$ and halves every iteration, and it stops once `rho` falls below a tolerance, `rhoend` $= 10^{-6}$. After 20 halvings, $2^{-20} \approx 9.5 \times 10^{-7}$ is below that tolerance. Both are defaults that `minimize` does not document, so we leave them alone; with them, a larger `maxiter` would not train any longer.

In [ ]:
best_per_iteration = []
evaluations = []


def record_loss(value):
    evaluations.append(float(value))
    # The optimizer first computes the loss at num_params + 1 points, then num_params + 4 more times per iteration;
    # the last num_params + 1 values of an iteration are the points it keeps.
    computed_after_start = len(evaluations) - (num_params + 1)
    if computed_after_start >= 0 and computed_after_start % (num_params + 4) == 0:
        best_per_iteration.append(min(evaluations[-(num_params + 1) :]))
        clear_output(wait=True)
        plt.figure(figsize=(6, 3.6))
        plt.plot(best_per_iteration, marker="o")
        plt.xlabel("iteration")
        plt.ylabel("loss")
        plt.show()


loss = make_loss(model, X_train, y_train)


def loss_with_plot(w):
    value = loss(w)
    jax.debug.callback(record_loss, value)
    return value


@jaspify(terminal_sampling=True)
def train():
    w0 = jnp.array([1.5] * num_params)
    result = minimize(loss_with_plot, w0, method="COBYLA", options={"maxiter": 20})
    return result.x, result.fun


w_opt, final_loss = train()
print(f"loss {best_per_iteration[0]:.3f} -> {float(final_loss):.3f} after {len(best_per_iteration) - 1} iterations")

## Checking the predictions

We confirm a low loss against the predictions themselves, on the test set as well as the training set. For a $\pm 1$ classifier the sign is what matters.

Do not be surprised if the training accuracy comes out *lower* than the test accuracy. Because we picked the training flowers to span each species from its shortest petal to its longest, the training set deliberately contains the hardest cases: the longest-petalled versicolor (5.1 cm) and the shortest-petalled virginica (4.5 cm) both sit inside the other species' range, where a boundary that suits the rest of the data is likely to misclassify them. With only eight training flowers, missing those two alone costs 25%. In the test set, the flowers from the overlap are a small share among many typical ones, so its accuracy is the better measure of the model.

In [ ]:
@jaspify(terminal_sampling=True)
def predict(X):
    return jnp.array([model(X[i], w_opt) for i in range(X.shape[0])])


for name, features, targets in [("training", X_train, y_train), ("test", X_test, y_test)]:
    correct = np.sign(np.asarray(predict(features))) == np.sign(np.asarray(targets))
    print(f"{name:8s} accuracy: {np.mean(correct):.3f}  ({int(np.sum(~correct))} of {len(correct)} misclassified)")

### The decision boundary

To see what the model has learned, we evaluate it not only on the flowers but on a grid of points covering the whole square of scaled features: petal length across, petal width up. At every grid point, the background colour shows the model's output $\langle Z_0 \rangle$ for a flower with those features.

In [ ]:
resolution = 12
grid_axis = np.linspace(0.0, np.pi, resolution)
grid = jnp.array([[a, b] for b in grid_axis for a in grid_axis])


@jaspify(terminal_sampling=True)
def predict_grid():
    return jnp.array([model(grid[i], w_opt) for i in range(grid.shape[0])])


surface = np.asarray(predict_grid()).reshape(resolution, resolution)

plt.figure(figsize=(5.2, 4.2))
plt.contourf(grid_axis, grid_axis, surface, levels=20, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="model output  (−1 versicolor, +1 virginica)")
plt.contour(grid_axis, grid_axis, surface, levels=[0.0], colors="black", linewidths=2)

training = np.asarray(X_train)
testing = np.asarray(X_test)
plt.scatter(training[:, 0], training[:, 1], c=colors, s=90, edgecolors="black", zorder=3)
plt.scatter(testing[:, 0], testing[:, 1], facecolors="none", edgecolors=test_colors, s=90, linewidths=2, zorder=3)
plt.xlabel("scaled petal length (angle $x_0$)")
plt.ylabel("scaled petal width (angle $x_1$)")
plt.title("decision boundary")
plt.tight_layout()
plt.show()

How to read the plot:

- **Background colour:** the model's output for a flower with those two features, read off the colour bar. Deep blue, close to $-1$, means the model says versicolor; deep red, close to $+1$, means virginica; the pale band in between is where the model is unsure.
- **Black line:** where the output crosses $0$. This is the **decision boundary**: the model calls everything on the blue side versicolor and everything on the red side virginica.
- **Dots:** the real flowers, coloured by their *true* species, blue for versicolor and red for virginica. Filled dots are the eight training flowers, hollow dots the test flowers.
- **A dot on the wrong side of the line is misclassified.** A red dot on the blue side is a virginica that the model calls versicolor, and a blue dot on the red side the reverse. Most of them sit close to the line, where the two species overlap.

Two details. The line looks jagged only because the grid is coarse, $12 \times 12$ points, and each output is a sampled estimate; the model's actual output is a smooth function of the features. And dots on the edges of the square are test flowers whose values were clipped to the training range during scaling.

## What we have built

That completes the pipeline from the figure at the top. We started from flower measurements in centimetres, scaled them into rotation angles, encoded each flower into a two-qubit state, transformed that state with a trainable ansatz, and read out one number, $\langle Z_0 \rangle$, whose sign is the prediction. Training adjusted the parameters $\underline{w}$ until those predictions matched the labels of eight flowers, and the trained model classifies most of the test flowers correctly, flowers it never saw during training.

## Saving and reusing the model

A trained model is the circuit, which is code, plus the numbers that training produced. `save_model` from `qrisp.qml` stores the parts of a model in a file, each under a name we choose, and `load_model` returns them under the same names. We save four parts: the encoding, the ansatz, the observable and the trained parameters.

For the encoding, we save one that does the scaling itself: given the range of each feature in the training data through `data_range`, it scales raw measurements exactly as `minmax_scale` did, including clipping values outside that range. The saved model therefore takes flower measurements directly in centimetres.

In [ ]:
model_file = Path(tempfile.gettempdir()) / "iris_model.pkl"
save_model(
    model_file,
    encoding=Encoding.angle(n_features=2, data_range=(X_raw.min(axis=0), X_raw.max(axis=0))),
    ansatz=ansatz,
    observable=Z(0),
    parameters=w_opt,
)

In a new session, we load the parts and put the circuit together exactly as before. Like any format based on Python's pickle, a model file can run code when it is loaded, so only load files from a source you trust. Below we classify two new flowers that are not in the dataset.

In [ ]:
saved = load_model(model_file)


def loaded_model(x_cm):
    def prepare():
        qf = QuantumFloat(num_qubits)
        saved["encoding"].apply(qf, x_cm)
        saved["ansatz"].apply(qf, saved["parameters"])
        return qf

    return expectation(prepare, saved["observable"], precision=0.05)()


# Two new flowers: (petal length, petal width) in centimetres.
new_flowers = jnp.array([[4.0, 1.2], [6.0, 2.2]])


@jaspify(terminal_sampling=True)
def predict_new():
    return jnp.array([loaded_model(new_flowers[i]) for i in range(new_flowers.shape[0])])


for flower, output in zip(np.asarray(new_flowers), np.asarray(predict_new())):
    species = "virginica" if output > 0 else "versicolor"
    print(f"petal {flower[0]} cm by {flower[1]} cm -> output {output:+.2f} -> {species}")

The circuit itself can also be written out in OpenQASM 3, a standard format that other quantum software reads. Here it is for the first new flower, with its encoded angles and the trained parameters filled in:

In [ ]:
qf = QuantumFloat(num_qubits)
saved["encoding"].apply(qf, new_flowers[0])
saved["ansatz"].apply(qf, saved["parameters"])
print(qf.qs.compile().to_qasm3())

## Swapping the parts

The model we saved has four parts: an encoding, an ansatz, an observable and the trained parameters. Because each one is a separate object, we can replace one part and keep the rest. We start from the model we loaded, `saved`, and change one part at a time. In the figure for each change, grey marks what we keep from the saved model and orange what is new.

A changed model has to be trained again. Training does not always have to start from scratch, though: when the new model still contains the old one, the trained parameters are a much better starting point than any fixed value.

The helper `make_model` builds a model from any encoding, ansatz and observable, in the same way as `loaded_model` above. `train_from` trains a model from a given starting point and returns the loss before and after training, together with the trained parameters. The saved encoding does its own scaling, so we train on the measurements in centimetres.

In [ ]:
def make_model(encoding, ansatz, observable):
    def model_fn(x, w):
        def prepare():
            qf = QuantumFloat(num_qubits)
            encoding.apply(qf, x)
            ansatz.apply(qf, w)
            return qf

        return expectation(prepare, observable, precision=0.05)()

    return model_fn


def train_from(model_fn, X, y, w0):
    @jaspify(terminal_sampling=True)
    def run():
        loss = make_loss(model_fn, X, y)
        result = minimize(loss, w0, method="COBYLA", options={"maxiter": 20})
        return loss(w0), result.fun, result.x

    return run()


X_cm, X_test_cm = jnp.asarray(X_raw), jnp.asarray(X_test_raw)

### A different ansatz

`Ansatz.hardware_efficient` takes the rotation axes as an argument. With `("y", "z")`, each qubit gets an $R_Y$ and then an $R_Z$ rotation in every layer. That is twice the parameters of `Ansatz.real_amplitudes`, and it gives the model access to complex amplitudes rather than only real ones.

<p align="center"><img src="../../_static/qml_swap_ansatz.svg" alt="The saved circuit with its ansatz replaced. The encoding R_Y(x0), R_Y(x1) and the readout Z0 are grey, kept from the saved model. The ansatz is orange, new: on each qubit R_Y then R_Z, a CX, then R_Y then R_Z again. The R_Y gates are grey and carry the trained angles w0 to w3; the R_Z gates are orange and start at angle 0." width="720"></p>

The new ansatz contains the old one. $R_Z(0)$ does nothing, so if we give the $R_Y$ gates the trained angles and set every $R_Z$ angle to $0$, the new model computes exactly what the saved one did. Training then starts from the trained model, and the extra rotations can only help it improve. In the parameter vector, each layer lists its $R_Y$ angles for both qubits first and then its $R_Z$ angles, so the trained $w_0, w_1$ go to positions 0 and 1, and $w_2, w_3$ to positions 4 and 5.

To see what this gains us, we also train the same model from our usual starting point, every angle at 1.5.

In [ ]:
ansatz_he = Ansatz.hardware_efficient(num_qubits=num_qubits, layers=1, rotations=("y", "z"))
model_he = make_model(saved["encoding"], ansatz_he, saved["observable"])

w = saved["parameters"]
w_reused = jnp.array([w[0], w[1], 0.0, 0.0, w[2], w[3], 0.0, 0.0])
w_fresh = jnp.array([1.5] * hardware_efficient_size(num_qubits, layers=1, rotations=("y", "z")))

for name, w0 in [("from the trained angles", w_reused), ("from 1.5", w_fresh)]:
    start, end, _ = train_from(model_he, X_cm, y_train, w0)
    print(f"{name:24s} loss {float(start):.3f} -> {float(end):.3f}")

Started from the trained angles, the new model begins at about the loss the saved model reached and improves on it a little. Started from 1.5, the same model stays at a much higher loss. It can represent the better solution, but the optimizer only searches near its starting point and does not find it.

### A different encoding

`Encoding.dense_angle` packs *two* features per qubit, one as a rotation angle and one as a relative phase, so two qubits carry four features instead of two. We keep the register the same size while giving the model all four Iris measurements, sepals as well as petals. Like the saved encoding, it does its own scaling, here from the range of all four measurements in the training data.

<p align="center"><img src="../../_static/qml_swap_encoding.svg" alt="The saved circuit with its encoding replaced. The encoding is orange, new: qubit 0 gets R_Y(x0) and a phase P(x1), qubit 1 gets R_Y(x2) and P(x3), where x0 and x1 are sepal length and width and x2 and x3 petal length and width. The ansatz R_Y(w0), R_Y(w1), CX, R_Y(w2), R_Y(w3) and the readout Z0 are grey: kept, but the ansatz parameters start again." width="720"></p>

This time the trained parameters are of little use. They were tuned for petal length on qubit 0 and petal width on qubit 1, but qubit 0 now carries the sepal measurements, and the petal width enters as a phase. The number of parameters still matches, so the code would accept them, but nothing makes them suited to the new input. We start again from 1.5.

In [ ]:
X4_cm = measurements[train_idx]
encoding_dense = Encoding.dense_angle(n_features=4, data_range=(X4_cm.min(axis=0), X4_cm.max(axis=0)))
model_dense = make_model(encoding_dense, saved["ansatz"], saved["observable"])

start, end, _ = train_from(model_dense, jnp.asarray(X4_cm), y_train, jnp.array([1.5] * num_params))
print(f"dense_angle, 4 features on {num_qubits} qubits: loss {float(start):.3f} -> {float(end):.3f}")

### A different readout

If we pass a *list* of observables, `expectation` returns one entry per observable instead of a single number. This is how a model with several outputs is read out, for instance one output per class. Here $\langle Z_0 \rangle$ keeps the meaning it was trained for, $+1$ for virginica, and the new output $\langle Z_1 \rangle$ gives $+1$ for versicolor.

A summed operator such as `Z(0) + Z(1)` would instead collapse both qubits into a single number, from which the individual contributions cannot be recovered.

<p align="center"><img src="../../_static/qml_swap_observable.svg" alt="The saved circuit with its readout replaced. The encoding and the ansatz, with its trained angles w0 to w3, are grey, kept. The readout is orange, new: qubit 0 is measured for Z0, +1 meaning virginica as trained, and qubit 1 for Z1, +1 meaning versicolor." width="720"></p>

The circuit is unchanged, so the trained parameters fit it exactly, and the first output is already trained. Starting from them, training only has to bring the second output into line without spoiling the first. We then check each output on the test set.

In [ ]:
# One output per species: (+1, -1) for virginica and (-1, +1) for versicolor.
y_two = jnp.stack([y_train, -y_train], axis=1)
model_two = make_model(saved["encoding"], saved["ansatz"], [Z(0), Z(1)])

start, end, w_two = train_from(model_two, X_cm, y_two, saved["parameters"])
print(f"two observables: loss {float(start):.3f} -> {float(end):.3f}")


@jaspify(terminal_sampling=True)
def predict_two():
    return jnp.array([model_two(X_test_cm[i], w_two) for i in range(X_test_cm.shape[0])])


outputs = np.asarray(predict_two())
for column, (observable, species, sign) in enumerate([("Z_0", "virginica", 1), ("Z_1", "versicolor", -1)]):
    accuracy = np.mean(np.sign(outputs[:, column]) == sign * np.asarray(y_test))
    print(f"<{observable}> (+1 for {species}): test accuracy {accuracy:.3f}")

The loss starts high only because $\langle Z_1 \rangle$ is not trained yet; $\langle Z_0 \rangle$ starts where the saved model left it. After training, both outputs classify most of the test flowers correctly.

**When can we reuse the trained parameters?** When the circuit is unchanged, as with the new readout, or the new model contains the old one, as with the larger ansatz, the trained parameters are the natural starting point, and training carries on from what was already learned. When the input to the circuit changes, as with the new encoding, they were tuned for different states, and we start again.

## Amplitude encoding

`Encoding.amplitude` writes the features into the *amplitudes* of the register rather than into rotation angles. A register of $n$ qubits holds $2^n$ amplitudes, so it encodes exponentially more features per qubit than angle encoding: 4 features on 2 qubits, 1024 on 10.

Two consequences follow from a quantum state having unit norm. The feature vector is normalized internally, so only its *direction* is encoded and its length is lost. And the vector is zero-padded up to the size of the register, since a register holds a power of two amplitudes while feature vectors rarely do.

In [ ]:
qf = QuantumFloat(2)
Encoding.amplitude(n_features=3).apply(qf, [1.0, 2.0, 3.0])

# Padded to (1, 2, 3, 0), then normalized by sqrt(14).
print("amplitudes:", np.round(np.array(qf.qs.statevector("array")).real, 4))
print("expected  :", np.round(np.array([1, 2, 3, 0]) / np.sqrt(14), 4))

Note that `statevector("array")` is indexed in reverse bit order, so the amplitude of $\ket{i}$ is not at index $i$ of that array. That is why the two rows above agree only up to that reordering.

## A curve one encoding layer cannot fit

So far every feature has entered the circuit once. That limits which functions the model can represent at all, whatever its parameters. We now build a task where the label depends on feature 0 alone: across $[0, \pi]$ it goes from $+1$ down to $-1$ and back up to $+1$, at five evenly spaced points. We hold feature 1 at $0$.

In [ ]:
X_wave = jnp.array([[k * np.pi / 4, 0.0] for k in range(5)])
y_wave = jnp.array([1.0, 0.0, -1.0, 0.0, 1.0])

for i in range(5):
    print(f"  feature 0 = {float(X_wave[i, 0]):.3f}  ->  label {float(y_wave[i]):+.0f}")

With one encoding layer, feature 0 is a single rotation angle, and the model output is a single sinusoid in it: $a + b\cos x_0 + c\sin x_0$. Outputting $+1$ at both $x_0 = 0$ and $x_0 = \pi$ forces $a = 1$ and $b = 0$, and an expectation value never exceeds $1$, which then forces $c = 0$ as well. The only sinusoid through both end points is the constant $+1$, so the model cannot reach $-1$ in between. No choice of parameters fits all five points, so the loss cannot get close to zero.

In [ ]:
@jaspify(terminal_sampling=True)
def train_single():
    loss = make_loss(model, X_wave, y_wave)
    return minimize(loss, jnp.array([0.5] * num_params), method="COBYLA", options={"maxiter": 20}).fun


single_loss = float(train_single())
print(f"one encoding layer, {num_params} parameters -> loss {single_loss:.3f}")

A picture shows why. Take a single qubit and, for the moment, set every angle of the ansatz $W$ to $0$, so that only the encoding acts. $R_Y(x_0)$ turns the qubit by $x_0$, and the output $\langle Z \rangle$ is the height of the arrow, as in the Bloch figure of the encoding section. As $x_0$ runs from $0$ to $\pi$, the arrow sweeps half a circle, from $\ket{0}$ at the top to $\ket{1}$ at the bottom: the output falls from $+1$ to $-1$ and cannot come back up.

Trained angles in $W$ do not change this. $W$ moves the whole half circle to another place on the Bloch sphere, but it stays a half circle, and the two ends of a half circle are opposite points: when one end reads $+1$, the other reads $-1$. This is the formula above, seen as a picture.

The remedy is to encode the feature more than once. If $R_Y(x_0)$ appears twice, the qubit turns by $x_0$ twice, so as $x_0$ runs from $0$ to $\pi$ the arrow sweeps the full circle and returns to where it started. With every angle of $W$ at $0$, the output is then $\cos 2x_0$, which passes through all five target points.

<p align="center"><img src="../../_static/qml_reuploading.svg" alt="Two panels, one qubit with every angle of W at 0, as feature x0 runs from 0 to pi. Left, one encoding layer, R_Y(x0) then W: on the Bloch circle the state sweeps half a circle from |0> at x0 = 0 to |1> at x0 = pi, and the output <Z> = cos(x0) falls from +1 to -1, missing the targets +1, 0, -1, 0, +1 at x0 = 0, pi/4, pi/2, 3pi/4, pi. Right, two encoding layers, R_Y(x0), W, R_Y(x0), W: the state sweeps the full circle, through |1> at x0 = pi/2 and back to |0> at x0 = pi, and the output <Z> = cos(2 x0) passes through all five targets." width="800"></p>

This is **data re-uploading**: the features are encoded again between parameterized layers, rather than only once at the start:

$$\prod_{l=1}^{L} W(\underline{w}^{(l)})\, S(\underline{x})$$

On two entangled qubits, the picture of a single arrow no longer applies, but the conclusion still holds: each further encoding layer adds higher frequencies of the features to the output. With $L$ layers, the output can contain $\cos(L x_0)$ and $\sin(L x_0)$, not only $\cos x_0$ and $\sin x_0$.

`Ansatz.data_reuploading` builds this circuit. It takes an encoding and an ansatz, so any encoding can be alternated with any ansatz. The per-layer ansatz determines how the parameter vector is split into one block per layer, and `num_parameters` reports the total.

In [ ]:
reupload_layers = 2
reuploading = Ansatz.data_reuploading(
    num_qubits=num_qubits,
    encoding=encoding,
    ansatz=Ansatz.real_amplitudes(num_qubits=num_qubits, layers=1),
    layers=reupload_layers,
)
total_params = reuploading.num_parameters


def circuit_reupload(x, w):
    qf = QuantumFloat(num_qubits)
    reuploading.apply(qf, w, x)
    return qf


def model_reupload(x, w):
    return expectation(lambda: circuit_reupload(x, w), Z(0), precision=0.05)()


@jaspify(terminal_sampling=True)
def train_reupload():
    loss = make_loss(model_reupload, X_wave, y_wave)
    return minimize(loss, jnp.array([0.5] * total_params), method="COBYLA", options={"maxiter": 20}).fun


reupload_loss = float(train_reupload())
print(f"{reupload_layers} encoding layers, {total_params} parameters -> loss {reupload_loss:.3f}")

With one encoding layer, the loss stays high, because the circuit cannot represent the curve. With two, the loss drops close to zero: the second encoding layer lets the output come back up.

In [ ]:
@jaspify(terminal_sampling=True)
def train_spsa():
    loss = make_loss(model, X_train, y_train)
    w0 = jnp.array([1.5] * num_params)
    result = minimize(loss, w0, method="SPSA", options={"maxiter": 60})
    return loss(w0), result.fun


before, after = train_spsa()
print(f"SPSA: loss {float(before):.3f} -> {float(after):.3f}")

## Summary

We wrote the loss as an ordinary function and trained with a single call to `minimize`. The encoding and the ansatz are configured objects, each independent of the others, which is why we could swap every stage without touching the rest:

| Stage | Primitives |
|---|---|
| Preprocessing | `minmax_scale`, `normalize`, `binarize` |
| Encoding | `Encoding.angle`, `Encoding.dense_angle`, `Encoding.amplitude`, `Encoding.basis` |
| Ansatz | `Ansatz.real_amplitudes`, `Ansatz.hardware_efficient`, `Ansatz.data_reuploading` |
| Measurement | `expectation` with one or several observables |
| Optimizer | `minimize` with `COBYLA` or `SPSA` |

The preprocessing functions exist because each encoding has an input contract: bounded angles for angle encoding, unit norm for amplitude encoding, strict bits for basis encoding. Anything beyond that (imputation, dimensionality reduction, categorical features) is ordinary classical data preparation, and the established tools do it better.

Four things are worth carrying forward:

**Features must be scaled before encoding.** Rotation angles are $2\pi$-periodic, and nothing warns you when a feature wraps around: the circuit runs and the model simply trains badly. `minmax_scale` fits on the training data and reuses that range everywhere else.

**Judge a model on data it was not trained on.** A low training loss says the model fits its training points; only the test set says it learned the classes.

**The optimizers are local.** Results depend on the starting point, so vary it before concluding anything about a model. When a changed model contains a trained one, start from the trained parameters.

**Tracing is not differentiation.** Jasp makes the circuit fast to run; it does not provide gradients through quantum execution. That is why training is gradient-free here. The parameter-shift rule computes gradients from extra circuit runs instead, as the [transfer learning tutorial](QMLTransferLearning.ipynb) shows.